# GCN for MDM2 Inhibitor Classification (Google Colab)

Binary classifier that predicts whether a molecule inhibits **MDM2** (target **CHEMBL5023**) from its SMILES string, using a **Graph Convolutional Network (GCN)** built with PyTorch Geometric.

**Paper being replicated:** *Machine Learning-Guided Discovery of Natural MDM2 Inhibitors: A Multistage In Silico Pipeline from Screening to ADMET Profiling* (Budha et al., Advanced Theory and Simulations 2026, DOI [10.1002/adts.202501502](https://doi.org/10.1002/adts.202501502)).

**What we replace:** the paper used a RandomForestClassifier on 2D descriptors for the screening/classification step. We substitute a GCN that operates directly on molecular graphs (atoms = nodes, bonds = edges), which captures topology that 2D fingerprints can miss. Everything downstream in the paper (docking, MD, DFT, ADMET) is unchanged and out of scope here.

**Task & label:** active (MDM2 inhibitor) iff `pChEMBL >= 6.0` (default threshold; ~1 uM potency), else inactive.

**Pipeline in this notebook:**

1. Environment check (GPU/CPU)
2. Install dependencies
3. Get the repo (clone or upload a zip)
4. Download ChEMBL MDM2 bioactivity data
5. Train the GCN
6. Evaluate on the held-out test split
7. Predict on a few example SMILES

Run the cells top to bottom. The whole run takes ~15-30 min on the free CPU runtime and ~5-10 min with a GPU runtime enabled (**Runtime > Change runtime type > GPU**).

In [1]:
# --- Environment check ---
import platform, sys
print("Python:", platform.python_version())
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: none (CPU runtime) -> training will use the CPU config")
try:
    import torch_geometric
    print("PyG:", torch_geometric.__version__)
except ImportError:
    print("PyG: not installed yet (next cell installs it)")
try:
    from rdkit import Chem
    print("RDKit:", Chem.rdBase.rdkitVersion)
except ImportError:
    print("RDKit: not installed yet")

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
PyG: not installed yet (next cell installs it)
RDKit: not installed yet


In [2]:
%%capture
# --- Install dependencies ---
# Colab already ships torch; installing torch_geometric pulls the right deps for it.
!pip install -q rdkit torch_geometric pandas numpy scikit-learn matplotlib seaborn requests tqdm pyyaml
# On a CPU-only runtime this installs (or keeps) a CPU torch wheel; on a GPU runtime
# it installs the CUDA-enabled wheel automatically.
!pip install -q torch

In [3]:
# --- Verify install ---
import torch_geometric, rdkit, sklearn, pandas, numpy
print("PyG", torch_geometric.__version__, "| RDKit", rdkit.__version__, "| sklearn", sklearn.__version__)

PyG 2.8.0.post1 | RDKit 2026.03.5 | sklearn 1.6.1


In [ ]:
# --- Get the repo (clone from GitHub) ---
import os, subprocess
REPO_URL = "https://github.com/Techbjd/gcn-code.git"
if os.path.isdir("gcn-code") and os.path.exists("gcn-code/src"):
    print("gcn-code already present.")
else:
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("Cloned.")

In [12]:
# --- cd into the repo root ---
import os
if os.path.isdir("/content/gcn-code") and os.path.exists("/content/gcn-code/src"):
    %cd /content/gcn-code
elif os.path.exists("/content/src"):
    %cd /content
else:
    print("WARNING: repo root not found. Re-run the clone/upload cell (zip must contain src/).")
%pwd
!ls -la

/content/gcn-code
total 68
drwxr-xr-x 9 root root  4096 Aug 16 08:50 .
drwxr-xr-x 1 root root  4096 Aug 16 08:50 ..
drwxr-xr-x 2 root root  4096 Aug 16 08:50 checkpoints
drwxr-xr-x 2 root root  4096 Aug 16 08:50 config
drwxr-xr-x 4 root root  4096 Aug 16 08:50 data
drwxr-xr-x 2 root root  4096 Aug 16 08:50 notebooks
drwxr-xr-x 3 root root  4096 Aug 16 08:50 outputs
-rw-r--r-- 1 root root 14939 Aug 16 08:50 README.md
-rw-r--r-- 1 root root   151 Aug 16 08:50 requirements.txt
drwxr-xr-x 2 root root  4096 Aug 16 08:50 scripts
-rw-r--r-- 1 root root  9109 Aug 16 08:50 SPEC.md
drwxr-xr-x 5 root root  4096 Aug 16 08:50 src


In [13]:
# --- Download ChEMBL MDM2 (CHEMBL5023) bioactivity data ---
# Skips if data/raw/chembl_mdm2.csv already exists (cached).
import os
os.makedirs("data/raw", exist_ok=True)
!python -m src.data.download_chembl --config config/cpu.yaml
print("---") if os.path.exists("data/raw/chembl_mdm2.csv") else print("no CSV yet")
!head -3 data/raw/chembl_mdm2.csv 2>/dev/null || true

<frozen runpy>:128: RuntimeWarning: 'src.data.download_chembl' found in sys.modules after import of package 'src.data', but prior to execution of 'src.data.download_chembl'; this may result in unpredictable behaviour
4927 rows ready at data/raw/chembl_mdm2.csv
---
molecule_chembl_id,smiles,standard_type,standard_value,standard_units,relation,pchembl_value
CHEMBL120563,CN(C)C(=S)SSC(=S)N(C)C,EC50,180.0,nM,=,6.75
CHEMBL1233798,O=C(O)c1[nH]c2cc(Cl)ccc2c1-c1c(-c2ccccc2)ncn1Cc1ccc(Cl)cc1,Ki,920.0,nM,=,6.04


In [14]:
# --- Train the GCN with the right device config ---
# GPU detected -> config/colab.yaml is a copy of gpu.yaml (device: cuda, with CPU
# fallback handled in src/utils.get_device). CPU runtime -> a copy of cpu.yaml.
import os, shutil, torch
if torch.cuda.is_available():
    shutil.copy("config/gpu.yaml", "config/colab.yaml")
    print("GPU detected -> config/colab.yaml (device: cuda)")
else:
    shutil.copy("config/cpu.yaml", "config/colab.yaml")
    print("CPU runtime -> config/colab.yaml (device: cpu)")
!python -m src.train --config config/colab.yaml

GPU detected -> config/colab.yaml (device: cuda)
Graphs: total=4927 train=3203 val=739 test=985 (active=4102, inactive=825, threshold=6.0)
Model: gcn (sum pooling), 210,946 params
Device: cuda | Train=3203 Val=739 Test=985
Epoch   1/150: loss=0.7730 acc=0.7299 | val_loss=0.6286 val_acc=0.1691 val_auc=0.6490
  EarlyStopping: val_loss improved to 0.6286 -> checkpoints/best_model.pt
Epoch   2/150: loss=0.4762 acc=0.7374 | val_loss=0.2879 val_acc=0.8038 val_auc=0.7810
  EarlyStopping: val_loss improved to 0.2879 -> checkpoints/best_model.pt
Epoch   3/150: loss=0.3636 acc=0.7786 | val_loss=0.3406 val_acc=0.5196 val_auc=0.8271
  EarlyStopping: no improvement (1/20)
Epoch   4/150: loss=0.2761 acc=0.7662 | val_loss=0.2138 val_acc=0.7767 val_auc=0.8777
  EarlyStopping: val_loss improved to 0.2138 -> checkpoints/best_model.pt
Epoch   5/150: loss=0.2530 acc=0.7877 | val_loss=0.2449 val_acc=0.8187 val_auc=0.8811
  EarlyStopping: no improvement (1/20)
Epoch   6/150: loss=0.2648 acc=0.8005 | val_los

In [15]:
# --- Evaluate the best checkpoint on the held-out test split ---
!python -m src.evaluate --config config/colab.yaml
print("--- outputs/test_metrics.json ---")
!cat outputs/test_metrics.json

Graphs: total=4927 train=3203 val=739 test=985 (active=4102, inactive=825, threshold=6.0)
Loaded checkpoint: checkpoints/best_model.pt | Device: cuda

=== TEST EVALUATION: 985 molecules ===
              precision    recall  f1-score   support

           0     0.5404    0.9167    0.6799       168
           1     0.9800    0.8397    0.9044       817

    accuracy                         0.8528       985
   macro avg     0.7602    0.8782    0.7922       985
weighted avg     0.9050    0.8528    0.8661       985


Saved test metrics -> outputs/test_metrics.json
--- outputs/test_metrics.json ---
{
  "accuracy": 0.8527918781725888,
  "roc_auc": 0.9366002214839424,
  "pr_auc": 0.986050623977517,
  "precision": 0.98,
  "recall": 0.8396572827417381,
  "f1": 0.9044166117336849,
  "confusion_matrix": [
    [
      154,
      14
    ],
    [
      131,
      686
    ]
  ]
}

In [16]:
# --- Predict activity for a few example SMILES ---
# Predictions sorted by probability descending; class=1 if prob >= 0.5.
!python -m src.predict --config config/colab.yaml \
    --smiles_file data/raw/example_smiles.csv \
    --output outputs/predictions_colab.csv
print("--- outputs/predictions_colab.csv ---")
!cat outputs/predictions_colab.csv

Featurized 5/5 molecules.
Predictions for 5 molecules written to outputs/predictions_colab.csv
--- outputs/predictions_colab.csv ---
id,smiles,probability_active,class
1,CC(=O)Oc1ccccc1C(=O)O,0.5200516581535339,1
0,C[C@@H]1CC[C@H]2C(=O)N(C(=O)N(c3ccccc3C)C2=O)c2ccccc21,0.4925931394100189,0
4,CC(=O)Nc1ccc(O)cc1,0.37164121866226196,0
3,CC(C)Cc1ccc(cc1)C(C)C(=O)O,0.3699416220188141,0
2,Cn1c(=O)c2c(ncn2C)n(C)c1=O,0.016628526151180267,0


In [ ]:
# --- Screen COCONUT with the trained GCN (~738k natural products) ---
# Downloads the COCONUT lite CSV once, then scores it in parallel chunks.
import os, zipfile, glob
if not os.path.exists("coconut_csv_lite.csv"):
    os.system("wget -q -O coconut.zip https://coconut.s3.uni-jena.de/prod/downloads/2026-08/coconut_csv_lite-08-2026.zip")
    with zipfile.ZipFile("coconut.zip") as z:
        z.extractall("coconut_data")
    os.rename(glob.glob("coconut_data/*.csv")[0], "coconut_csv_lite.csv")
    print("COCONUT CSV ready.")
!python -m src.screen --config config/colab.yaml \
    --input coconut_csv_lite.csv \
    --smiles_col canonical_smiles --id_col id \
    --output outputs/coconut_predictions.csv \
    --chunk_size 50000

In [ ]:
# --- Summarize hits + build a diverse docking set ---
!python -m src.analyze_hits --predictions outputs/coconut_predictions.csv \
    --library coconut_csv_lite.csv \
    --id_col id --smiles_col canonical_smiles \
    --n_diverse 200 --outdir outputs

# Results & Next Steps

**What you should see:**

- Per-epoch training lines with train loss/acc and val loss/acc/auc, early stopping on val loss.
- `outputs/test_metrics.json` with accuracy, ROC-AUC, PR-AUC, precision, recall, F1.
- `outputs/plots/{roc_curve,pr_curve,confusion_matrix}.png` and `outputs/training_history.csv`.
- `outputs/predictions_colab.csv` ranking the example molecules by predicted inhibition probability.

**COCONUT screening (large library):**

1. `src.screen` downloads the COCONUT lite CSV once, then scores all ~738k natural products in parallel chunks. Output: `outputs/coconut_predictions.csv` (ranked by `probability_active`).
2. `src.analyze_hits` summarizes the screen, dedups structural duplicates, and keeps one molecule per Murcko scaffold to produce a diverse docking set: `outputs/hits.csv` (all hits) and `outputs/diverse_hits.csv` / `.smi` (diverse set, ready for docking).

**Next steps (following the paper's downstream stages):**

1. **Docking** the diverse hits against MDM2 (e.g., AutoDock Vina/Glide) to filter by binding pose and score.
2. **Molecular dynamics (MD)** on the best complexes to confirm stability.
3. **DFT / ADMET profiling** of the surviving candidates.

**Interpretation caveats:** the GCN is a screening prior, not a docking oracle — a high score means the model believes the molecule's graph resembles known MDM2 inhibitors, not that it binds. Many COCONUT molecules score exactly 1.0 (probability saturation); do not rely on ordering within that tier — the scaffold-diverse set is the right thing to dock. Validate computationally (docking/MD) and experimentally before drawing conclusions. Research use only.